# Ghost 3D Forge — SF3D no Google Colab
Este notebook usa **Stable Fast 3D (SF3D)** para gerar um arquivo `.glb` a partir de uma imagem. Ative GPU no Colab antes de executar.

**Importante:** o SF3D oficial é *single-image*. Para começar, use a vista de frente mais limpa. Frente/verso/laterais continuam úteis para conferência e para uma futura etapa multi-view.


In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())


## 1) Instalar o SF3D


In [ ]:
!apt-get -qq update
!apt-get -qq install -y git build-essential libgl1 libglib2.0-0
!pip -q install -U setuptools==69.5.1 wheel
!git clone --depth 1 https://github.com/Stability-AI/stable-fast-3d.git
%cd stable-fast-3d
!pip -q install -r requirements.txt
!pip -q install huggingface_hub


## 2) Login no Hugging Face
O modelo oficial é gated. Solicite acesso ao modelo no Hugging Face e cole abaixo um token com permissão de leitura.


In [ ]:
from huggingface_hub import login
from getpass import getpass
HF_TOKEN = getpass('Cole seu token do Hugging Face: ')
login(token=HF_TOKEN)


## 3) Enviar imagem
Use uma imagem PNG/JPG com o objeto centralizado. Fundo transparente ou branco costuma funcionar melhor.


In [ ]:
from google.colab import files
uploaded = files.upload()
input_name = next(iter(uploaded.keys()))
print('Imagem:', input_name)


## 4) Gerar GLB
O preset abaixo usa textura 1024 e remesh triangular para equilibrar qualidade e memória.


In [ ]:
import os, shutil, glob, subprocess, sys
out_dir = '/content/ghost3d_output'
shutil.rmtree(out_dir, ignore_errors=True)
os.makedirs(out_dir, exist_ok=True)
cmd = [sys.executable, 'run.py', input_name, '--output-dir', out_dir, '--texture-resolution', '1024', '--remesh_option', 'triangle']
print('Executando:', ' '.join(cmd))
subprocess.run(cmd, check=True)
glbs = glob.glob(out_dir + '/**/*.glb', recursive=True)
print('GLBs encontrados:', glbs)


## 5) Baixar o GLB


In [ ]:
from google.colab import files
if not glbs:
    raise FileNotFoundError('Nenhum GLB foi gerado.')
final_glb = glbs[0]
files.download(final_glb)


## Próximo passo do Ghost 3D Forge
Depois que este fluxo estiver funcionando de forma estável, o site pode ganhar um modo **IA/Colab** separado do modo Visual Hull local. O Colab continua sendo temporário: quando a sessão desligar, precisa iniciar novamente.
